In [0]:
import requests
import json
from datetime import datetime



# CONFIGURACIÓN


APIFY_TOKEN = "apify_api_wX1yjVUvYU9cO6s9IrVIYdHmL4L3In0TjwRR"

ACTOR_ID = "atomus~tiktok-scraper"

URL = (
    f"https://api.apify.com/v2/acts/"
    f"{ACTOR_ID}/run-sync-get-dataset-items"
)



# PATH DE LANDING EN DATABRICKS


LANDING_PATH = "/Volumes/tiktok_data_eng/landing/archivos"



# HASHTAGS DEL PROYECTO


HASHTAGS = [
    "dataengineering",
    "ingenieriadedatos",
    "databricks",
    "pyspark",
    "dataengineer",
    "data"
]



# CANTIDAD DE RESULTADOS


RESULTS_PER_PAGE = 5000



# CONTENEDOR DE DATOS RAW


datos_raw = []



# INGESTA


print("=" * 70)
print("INICIANDO INGESTA DE TIKTOK")
print("=" * 70)

for i, hashtag in enumerate(HASHTAGS, start=1):

    print(f"\n[{i}/{len(HASHTAGS)}] Consultando #{hashtag}...")

    payload = {
        "hashtags": [
            hashtag
        ],
        "resultsPerPage": RESULTS_PER_PAGE,
        "searchSection": "video",
        "includeAuthorDetails": True
    }

    response = requests.post(
        URL,
        params={
            "token": APIFY_TOKEN
        },
        json=payload,
        timeout=300
    )

    print("Status:", response.status_code)

    if response.status_code not in [200, 201]:

        print(f"❌ Error consultando #{hashtag}")
        print(response.text)

        continue

    resultados = response.json()

    print(
        f"✅ {len(resultados)} registros recibidos"
    )

    # Guardamos exactamente lo que devuelve Apify
    datos_raw.extend(resultados)



# RESUMEN


print("\n" + "=" * 70)
print("RESUMEN")
print("=" * 70)

print(f"Hashtags consultados: {len(HASHTAGS)}")
print(f"Registros obtenidos: {len(datos_raw)}")



# GENERAR NOMBRE DEL ARCHIVO


timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

archivo = (
    f"{LANDING_PATH}/"
    f"tiktok_raw_{timestamp}.json"
)



# PERSISTENCIA EN VOLUME


with open(
    archivo,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        datos_raw,
        f,
        ensure_ascii=False,
        indent=2
    )



# RESULTADO


print("\n" + "=" * 70)
print("INGESTA FINALIZADA")
print("=" * 70)

print(f"Archivo generado:")
print(archivo)

print(f"\nRegistros almacenados: {len(datos_raw)}")